In [ ]:
import numpy as np
# from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


import bisect

import essrs

In [ ]:
N_SIMULATIONS: int = 10

In [ ]:
# print(help(gaussian_kde))

In [ ]:
def neals_funnel(x: float, y: float) -> float:
    return 1. / (2 * np.pi) * np.exp(-0.5 * y**2) * np.exp(-y) * np.exp(-0.5 * (x / np.exp(y))**2)

In [ ]:
N_BINS = 100

xe = np.linspace(-20, 20, N_BINS+1)
ye = np.linspace(-4, 4, N_BINS+1)

xc = (xe[:-1] + xe[1:]) / 2.0
yc = (ye[:-1] + ye[1:]) / 2.0

dx = xc[1] - xc[0]
dy = yc[1] - yc[0]

gx, gy = np.meshgrid(xc, yc)

gz = neals_funnel(gx, gy)
gz /= np.sum(gz) * (dx * dy)

In [ ]:
def enclosed_probability(sigma: int, df: int) -> float:
    return 1 - np.exp(-(sigma**2) / df)

SIGMA_1 = enclosed_probability(1, 2)
SIGMA_2 = enclosed_probability(2, 2)
SIGMA_3 = enclosed_probability(3, 2)

print(SIGMA_1)
print(SIGMA_2)
print(SIGMA_3)

def estimate_enclosed_region(Z, target_sum):
    z = np.sort(Z.ravel(), descending=True)
    c = np.cumsum(z)
    i = bisect.bisect(c, target_sum)

    M = np.zeros_like(Z)
    M[z[i] < Z] = 1

    return M

In [ ]:
results = []

for _ in range(N_SIMULATIONS):
    ics = np.random.normal(loc=[0.0, 0.0], scale=0.1, size=(100,2))

    fit = essrs.toy.toy_neals_funnel(
        max_n_steps=3000,        # max number of steps to run the sampler
        n_burn_in=2000,          # number of burn in steps to discard prior to main sampling stage
        initial_conditions=ics, # initial conditions
    )

    results.append(fit)

In [ ]:
def three_contours(Z: np.ndarray) -> np.ndarray:
    M = estimate_enclosed_region(Z, SIGMA_1)
    M += estimate_enclosed_region(Z, SIGMA_2)
    M += estimate_enclosed_region(Z, SIGMA_3)

    return M

In [ ]:
# self._fig = plt.figure(1)
#         self._gs = gridspec.GridSpec(
#             nrows=self._shape[0],
#             ncols=self._shape[1],
#             wspace=axis_wspace,
#             hspace=axis_hspace,
#         )

fig = plt.figure(1)
gs = gridspec.GridSpec(nrows=2, ncols=2)

ax_x = plt.subplot(gs[0,0])
ax_xy = plt.subplot(gs[1, 0])
ax_y = plt.subplot(gs[1,1])

ax_xy.contour(gx, gy, three_contours(gz*dx*dy), cmap="Reds", zorder=3)# cmap="spring")

z_marg_y = np.sum(gz, axis=0)
z_marg_y /= np.sum(z_marg_y) * dx
ax_x.plot(gx[0,:], z_marg_y, color="r")

z_marg_x = np.sum(gz, axis=1)
z_marg_x /= np.sum(z_marg_x) * dy
ax_y.plot(gy[:,0], z_marg_x, color="r")


for i in range(N_SIMULATIONS):
    Z, _, _ = np.histogram2d(
        x=results[i][:,:,0].ravel(),
        y=results[i][:,:,1].ravel(),
        bins=[xe, ye],
        density=True
    )
    Z = Z.T
    ax_xy.contour(gx, gy, three_contours(Z*dx*dy), cmap="Blues", zorder=1)

    ax_x.hist(results[i][:,:,0].ravel(), density=True, histtype="step", bins=xe, color="b", alpha=0.1)
    ax_y.hist(results[i][:,:,1].ravel(), density=True, histtype="step", bins=ye, color="b", alpha=0.1)



ax_xy.set_xlabel(r"$x$")
ax_xy.set_ylabel(r"$y$")
ax_xy.set_xlim(-15, 15)
ax_x.set_xlim(-15, 15)


plt.show()